# 🎬 YouTube 上传工具

> 只需运行一次授权，之后自动上传

---


In [ ]:
# 安装依赖
!pip install google-api-python-client google-auth-oauthlib requests pillow moviepy -q

In [ ]:
# =============================================
# 第一次运行请执行此单元格完成授权
# 只需做一次，以后不需要重复
# =============================================
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
import google.auth.exceptions

# 需要先创建 OAuth 凭据（见下方说明）
CLIENT_SECRETS_FILE = "client_secret.json"

SCOPES = ["https://www.googleapis.com/auth/youtube.upload"]

def get_authenticated_service():
    flow = InstalledAppFlow.from_client_secrets_file(
        CLIENT_SECRETS_FILE, SCOPES)
    credentials = flow.run_local_server(port=0)
    return build('youtube', 'v3', credentials=credentials)

try:
    youtube = get_authenticated_service()
    print("✅ YouTube 授权成功！")
except google.auth.exceptions.MSMismatchError:
    print("❌ 授权失败，请确保 client_secret.json 文件存在")
except Exception as e:
    print(f"⚠️ 需要先设置凭据：{e}")

---

## 📋 第一次使用：创建 Google Cloud OAuth 凭据

1. 用代理/VPN 打开 **https://console.cloud.google.com**
2. 新建项目 → 搜索启用 **YouTube Data API v3**
3. **API和服务** → **凭据** → **创建凭据** → **OAuth客户端ID**
4. 应用类型选 **桌面应用**
5. 下载 JSON → 重命名为 `client_secret.json`
6. 上传文件到 Colab 左侧文件区

> 💡 **如果打不开 cloud.google.com**：
> 用手机开 VPN 访问一次，以后在这个浏览器就能用了

---


In [ ]:
# 上传视频（执行此单元格）
import os
from googleapiclient.http import MediaFileUpload

VIDEO_PATH = "/root/.openclaw/workspace/shorts/shorts-2026-07-19.mp4"  # ← 改这里
TITLE = "🔥 在线游戏合集 | 免费玩 浏览器直接开"
DESCRIPTION = """🎮 免费在线游戏合集，无需下载，浏览器直接玩！

恐龙跑酷 / 宝石消消乐 / 水果忍者 / 弹珠台 / 篮球投篮 / 深渊幸存者

🌐 完整游戏列表: https://nima54851.github.io/game-platform/

#游戏 #HTML5 #在线游戏 #益智游戏 #休闲游戏 #Gaming"""
TAGS = ["在线游戏", "HTML5游戏", "益智游戏", "休闲游戏", "免费游戏"]
CATEGORY_ID = "20"  # Gaming
PRIVACY_STATUS = "public"

if not os.path.exists(VIDEO_PATH):
    from google.colab import files
    print("📂 请先上传视频文件到 Colab")
    uploaded = files.upload()
    VIDEO_PATH = list(uploaded.keys())[0]

request = youtube.videos().insert(
    part="snippet,status",
    body={
        "snippet": {
            "title": TITLE,
            "description": DESCRIPTION,
            "tags": TAGS,
            "categoryId": CATEGORY_ID
        },
        "status": {
            "privacyStatus": PRIVACY_STATUS
        }
    },
    media_body=MediaFileUpload(VIDEO_PATH, chunksize=-1, resumable=True)
)

print("📤 上传中，请耐心等待...")
response = None
while response is None:
    status, response = request.next_chunk()
    if status:
        print(f"  进度: {int(status.get('progress', 0) * 100)}%")

print(f"✅ 上传成功！")
video_id = response.get('id')
print(f"🔗 https://www.youtube.com/watch?v={video_id}")

---

## ⚡ 快速方式：上传当天自动生成的 Shorts

如果视频在服务器上，可以直接下载后再上传：

```python
# 从服务器下载最新视频
import urllib.request
url = "http://YOUR_SERVER_IP:PORT/shorts/shorts-2026-07-19.mp4"
urllib.request.urlretrieve(url, "shorts-today.mp4")
VIDEO_PATH = "shorts-today.mp4"
```
